In [41]:
import pandas as pd
import glob, os, re

# REQUIRED: choose context & genotype
context = "CG"     # e.g., "CG", "CHG", "CHH"
mutant  = "col"    # e.g., "met", "rdd", etc.

# 1. Point to your directory
gff_dir = "../gffs_by_clst_genotype"

# 2. Build a pattern that uses your variables
#    e.g. match: clst*.col.CG.w100.gff
pattern  = f"clst*.{mutant}.{context}.w100.gff"
gff_paths = glob.glob(os.path.join(gff_dir, pattern))


gff_paths.sort()
gff_paths

['../gffs_by_clst_genotype/clst0.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst1.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst10.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst11.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst12.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst13.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst14.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst15.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst16.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst2.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst3.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst4.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst5.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst6.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst7.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst8.col.CG.w100.gff',
 '../gffs_by_clst_genotype/clst9.col.CG.w100.gff']

In [42]:
# columns to read
usecols  = [0, 3, 4, 5, 8]
colnames = ["chr", "start", "end", "score", "attributes"]

dfs = []
for path in gff_paths:
    # 1) read only the needed columns
    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        header=None,
        usecols=usecols,
        names=colnames,
        dtype={
            "chr":      str,
            "start":      int,
            "end":        int,
            "score":      float,
            "attributes": str
        }
    )

    # 2) pull cluster ID from the filename
    basename = os.path.basename(path)
    m = re.search(r"clst(\d+)", basename)
    if not m:
        raise ValueError(f"No cluster ID found in {basename}")
    cluster = int(m.group(1))
    df["cluster"] = cluster

    # 3) parse seqid → chr 
    df["chr"] = (
        df["chr"]
          .str.replace(r"chr", "", regex=True)
    )

    # 4) split out c, t, n
    df[["c","t","n"]] = (
        df["attributes"]
          .str.extract(r"c=(\d+);t=(\d+);n=(\d+)")
          .astype(int)
    )

    # 5) keep exactly the columns you want, *including* cluster
    dfs.append(df[["cluster","chr","start","end","score","c","t","n"]])

# 6) merge into one big DataFrame
all_gffs = pd.concat(dfs, ignore_index=True)


In [43]:
all_gffs[all_gffs['cluster']==0]

,cluster,chr,start,end,score,c,t,n
0,0,1,101,200,0.8951,350,41,6
1,0,1,301,400,0.5487,62,51,2
2,0,1,401,500,0.8246,47,10,1
3,0,1,501,600,0.7206,98,38,3
4,0,1,601,700,0.8982,203,23,6
...,...,...,...,...,...,...,...,...
970797,0,L,48001,48100,0.4706,8,9,8
970798,0,L,48101,48200,0.1923,5,21,8
970799,0,L,48201,48300,0.0606,2,31,11
970800,0,L,48301,48400,0.0000,0,17,7


In [44]:
# keep only rows where 'chr' is exactly "1"–"5"
mask = all_gffs["chr"].str.match(r"^[1-5]$")
gffs = all_gffs[mask]
gffs


,cluster,chr,start,end,score,c,t,n
0,0,1,101,200,0.8951,350,41,6
1,0,1,301,400,0.5487,62,51,2
2,0,1,401,500,0.8246,47,10,1
3,0,1,501,600,0.7206,98,38,3
4,0,1,601,700,0.8982,203,23,6
...,...,...,...,...,...,...,...,...
16297919,9,5,26974801,26974900,0.8000,32,8,10
16297920,9,5,26974901,26975000,1.0000,4,0,2
16297921,9,5,26975101,26975200,1.0000,4,0,2
16297922,9,5,26975201,26975300,0.9773,43,1,14


In [45]:
# # sort by cluster first, then chr (both ascending)
# gffs_sorted = gffs.sort_values(by=["cluster", "chr"]).reset_index(drop=True)
# gffs_sorted


In [13]:
mutant

'col'

In [46]:
out_path = f"./data/master_{mutant}.{context}.chr1-5.fast.tsv"
gffs_sorted.to_csv(out_path, sep="\t", index=False)
print(f"Wrote: {out_path}")


Wrote: ./data/master_col.CG.chr1-5.fast.tsv


In [47]:
# 1. Show memory per column (including object-dtypes like strings)
mem_per_col = gffs.memory_usage(index=True, deep=True)
print(mem_per_col)

# 2. Sum it up for the total footprint
total = mem_per_col.sum()
print(f"\nTotal memory usage: {total/1024**2:.2f} MB")


Index      130068704
cluster    130068704
chr        942998104
start      130068704
end        130068704
score      130068704
c          130068704
t          130068704
n          130068704
dtype: int64

Total memory usage: 1891.66 MB


In [48]:
# 1) Downcast integers
for col in ["cluster", "start", "end", "c", "t", "n"]:
    gffs[col] = pd.to_numeric(gffs[col], downcast="unsigned")

# 2) Downcast floats
gffs["score"] = pd.to_numeric(gffs["score"], downcast="float")

# 3) Convert chr to categorical (or uint8 if you prefer)
# Option A: category
# all_gffs["chr"] = all_gffs["chr"].astype("category")

# Option B: numeric uint8
gffs["chr"] = pd.to_numeric(gffs["chr"], downcast="unsigned")

# 4) Check new memory usage
mem = gffs.memory_usage(deep=True).sum() / 1024**2
print(f"New total memory: {mem:.2f} MB")
gffs_sorted.info(memory_usage="deep")


/tmp/ipykernel_3749499/2920396641.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gffs[col] = pd.to_numeric(gffs[col], downcast="unsigned")
/tmp/ipykernel_3749499/2920396641.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gffs["score"] = pd.to_numeric(gffs["score"], downcast="float")


New total memory: 480.67 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16258588 entries, 0 to 16258587
Data columns (total 8 columns):
 #   Column   Dtype  
---  ------   -----  
 0   cluster  uint8  
 1   chr      uint8  
 2   start    uint32 
 3   end      uint32 
 4   score    float32
 5   c        uint32 
 6   t        uint32 
 7   n        uint8  
dtypes: float32(1), uint32(4), uint8(3)
memory usage: 356.6 MB


/tmp/ipykernel_3749499/2920396641.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gffs["chr"] = pd.to_numeric(gffs["chr"], downcast="unsigned")


In [54]:
gffs.reset_index()

,index,cluster,chr,start,end,score,c,t,n
0,0,0,1,101,200,0.8951,350,41,6
1,1,0,1,301,400,0.5487,62,51,2
2,2,0,1,401,500,0.8246,47,10,1
3,3,0,1,501,600,0.7206,98,38,3
4,4,0,1,601,700,0.8982,203,23,6
...,...,...,...,...,...,...,...,...,...
16258583,16297919,9,5,26974801,26974900,0.8000,32,8,10
16258584,16297920,9,5,26974901,26975000,1.0000,4,0,2
16258585,16297921,9,5,26975101,26975200,1.0000,4,0,2
16258586,16297922,9,5,26975201,26975300,0.9773,43,1,14


In [55]:
# 4) Check new memory usage
mem = gffs.memory_usage(deep=True).sum() / 1024**2
print(f"New total memory: {mem:.2f} MB")
gffs.info(memory_usage="deep")


New total memory: 480.67 MB
<class 'pandas.core.frame.DataFrame'>
Index: 16258588 entries, 0 to 16297923
Data columns (total 8 columns):
 #   Column   Dtype  
---  ------   -----  
 0   cluster  uint8  
 1   chr      uint8  
 2   start    uint32 
 3   end      uint32 
 4   score    float32
 5   c        uint32 
 6   t        uint32 
 7   n        uint8  
dtypes: float32(1), uint32(4), uint8(3)
memory usage: 480.7 MB


In [53]:
# 1. Show memory per column (including object-dtypes like strings)
mem_per_col = gffs.memory_usage(index=True, deep=True)
print(mem_per_col)

# 2. Sum it up for the total footprint
total = mem_per_col.sum()
print(f"\nTotal memory usage: {total/1024**2:.2f} MB")


Index      130068704
cluster     16258588
chr         16258588
start       65034352
end         65034352
score       65034352
c           65034352
t           65034352
n           16258588
dtype: int64

Total memory usage: 480.67 MB


In [23]:
out_path = f"./data/master_{mutant}.{context}.chr1-5.fast.tsv"
gffs_sorted.to_csv(out_path, sep="\t", index=False)
print(f"Wrote: {out_path}")


Wrote: ./data/master_col.CG.chr1-5.fast.tsv


In [26]:
%pwd

'/ceph/MethDev/JW240627--at-snmCT_with_TE/mCT_with_TE/kay'

In [56]:
# write in most effiecient data size
gffs.to_pickle("./data/gffs_sorted.pkl")

# # later, reload with the exact same dtypes preserved:
# all_gffs = pd.read_pickle("./data/gffs_sorted.pkl")


In [ ]:

# # 4. Concatenate into one big DataFrame
# all_gffs = pd.concat(dfs, ignore_index=True)

# # 5. (Optional) parse out a    ewtributes into columns
# def parse_attributes(attr_str):
#     return dict(item.split("=",1) for item in attr_str.strip().split(";") if "=" in item)

# attr_df = all_gffs["attributes"].apply(parse_attributes).apply(pd.Series)
# all_gffs = pd.concat([all_gffs.drop("attributes",1), attr_df], axis=1)

# # now `all_gffs` has one row per feature, with all your GFFs together!
# all_gffs.head()
